## TL;DR

In this notebook, we tested retrieval on five randomly sampled DocFinQA rows using ChromaDB
(all-MiniLM-L6-v2) with 500-character chunks and Gemini 2.5 Flash for
generation. Documents range from 270K to 1.4M characters (entire SEC
filings), producing 680–3,600 chunks each. Retrieval coverage was consistently below 1% of the source document, and
the retrieved chunks rarely contained the information needed to answer
the question.

Three failure modes
emerged: (1) fixed-width chunking splits tables across boundaries,
separating headers from values; (2) the embedding model matches on
topical keywords without distinguishing fiscal year or business segment,
so a question about 2003 retrieves 2012 data; and (3) repeated
structural patterns across years and subsidiaries make many chunks
near-equidistant in embedding space, rendering top-k selection
effectively arbitrary. Only one of five samples produced a correct
answer. Addressing this likely requires structure-aware chunking, hybrid
sparse/dense retrieval, or bypassing retrieval entirely in favor of
long-context models.

# DocFinQA Retrieval + Generation Diagnostic

Quick notebook to investigate the retrieval issue with DocFinQA: retrieved context is either the entire document or nothing at all. We test 5 random rows using ChromaDB for retrieval and Gemini 2.5 Flash for generation.

In [4]:
%%capture
# Install dependencies (uncomment if needed)
!pip install chromadb google-genai datasets

In [5]:
import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

Enter your Gemini API key:  ········


In [6]:
from datasets import load_dataset
import random

ds = load_dataset("kensho/DocFinQA", split="train", streaming=True)

# Collect a manageable pool, then sample from it
pool = []
for i, row in enumerate(ds):
    pool.append(row)
    if i >= 99:  # grab first 100 rows
        break

print(f"Loaded {len(pool)} rows")
print(f"Columns: {list(pool[0].keys())}")

random.seed(42)
samples = random.sample(pool, 5)

for i, s in enumerate(samples):
    print(f"\nSample {i+1}:")
    for k, v in s.items():
        if isinstance(v, str):
            print(f"  {k}: {len(v)} chars")
        else:
            print(f"  {k}: {v}")

Loaded 100 rows
Columns: ['Context', 'Question', 'Program', 'Answer']

Sample 1:
  Context: 555451 chars
  Question: 73 chars
  Program: 0 chars
  Answer: 14 chars

Sample 2:
  Context: 271894 chars
  Question: 118 chars
  Program: 155 chars
  Answer: 5 chars

Sample 3:
  Context: 1440541 chars
  Question: 47 chars
  Program: 167 chars
  Answer: 5 chars

Sample 4:
  Context: 397044 chars
  Question: 131 chars
  Program: 0 chars
  Answer: 4 chars

Sample 5:
  Context: 372954 chars
  Question: 75 chars
  Program: 140 chars
  Answer: 4 chars


In [7]:
# Inspect a single row to understand the document structure
row = samples[0]
print("=" * 80)
for k, v in row.items():
    print(f"\n--- {k} ---")
    if isinstance(v, str):
        print(f"Length: {len(v)} chars / ~{len(v.split())} words")
        print(v[:1000])
        if len(v) > 1000:
            print(f"\n... [{len(v) - 1000} more chars] ...")
    else:
        print(v)


--- Context ---
Length: 555451 chars / ~85044 words
Table	of	Contents

# UNITED	STATES

# SECURITIES	AND	EXCHANGE	COMMISSION

# ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES

EXCHANGE	ACT	OF	1934

For	the	fiscal	year	ended	December	31,	2008

# TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES

For	the	transition	period	from										to

Commission	File	number	1-7221

# MOTOROLA,	INC.

(Exact	name	of	registrant	as	specified	in	its	charter)

DELAWARE

(State	of	Incorporation)

(I.R.S.	Employer	Identification	No.)

1303	East	Algonquin	Road,	Schaumburg,	Illinois	60196

(Address	of	principal	executive	offices)

(Registrant's	telephone	number)

Securities	registered	pursuant	to	Section	12(b)	of	the	Act:

Title	of	Each	Class

# Common	Stock,	$3	Par	Value	per	Share

Name	of	Each	Exchange	on	Which	Registered

New	York	Stock	Exchange

# Chicago	Stock	Exchange

# Securities	registered	pursuant	to	Section	12(g)	of	the	Act:

Indicate	by	check	mark	if	the	registr

In [8]:
print(f"Working with {len(samples)} samples\n")
for i, s in enumerate(samples):
    print(f"Sample {i+1}:")
    for k, v in s.items():
        if isinstance(v, str):
            print(f"  {k}: {len(v)} chars")
        else:
            print(f"  {k}: {v}")
    print()

Working with 5 samples

Sample 1:
  Context: 555451 chars
  Question: 73 chars
  Program: 0 chars
  Answer: 14 chars

Sample 2:
  Context: 271894 chars
  Question: 118 chars
  Program: 155 chars
  Answer: 5 chars

Sample 3:
  Context: 1440541 chars
  Question: 47 chars
  Program: 167 chars
  Answer: 5 chars

Sample 4:
  Context: 397044 chars
  Question: 131 chars
  Program: 0 chars
  Answer: 4 chars

Sample 5:
  Context: 372954 chars
  Question: 75 chars
  Program: 140 chars
  Answer: 4 chars



## Chunking & Embedding into ChromaDB

We chunk each document and embed into a per-document ChromaDB collection, then retrieve the top-k chunks for each question.

In [11]:
import chromadb


def chunk_text(text, chunk_size=500, overlap=100):
    """Split text into overlapping chunks by character count."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


# --- IMPORTANT ---
# Adapt the field name below after inspecting the schema in the cells above.
# It might be "context", "document", "doc", "text", "filing", etc.
CONTEXT_FIELD = "Context"
QUESTION_FIELD = "Question"
ANSWER_FIELD = "Answer"

client = chromadb.Client()  # ephemeral in-memory client

TOP_K = 5
retrieval_results = []

for i, s in enumerate(samples):
    doc_text = str(s[CONTEXT_FIELD])
    question = str(s[QUESTION_FIELD])
    gold_answer = str(s[ANSWER_FIELD])

    # Chunk the document
    chunks = chunk_text(doc_text, chunk_size=500, overlap=100)
    print(f"\n{'=' * 80}")
    print(f"Sample {i+1}: {len(chunks)} chunks from {len(doc_text)} chars")
    print(f"Question: {question}")
    print(f"Gold answer: {gold_answer}")

    # Create a fresh collection per document
    col_name = f"doc_{i}"
    try:
        client.delete_collection(col_name)
    except Exception:
        pass
    collection = client.create_collection(name=col_name)

    # Add chunks
    collection.add(
        documents=chunks,
        ids=[f"chunk_{j}" for j in range(len(chunks))],
        metadatas=[{"chunk_idx": j, "start_char": j * 400} for j in range(len(chunks))],
    )

    # Query
    results = collection.query(query_texts=[question], n_results=TOP_K)

    retrieved_docs = results["documents"][0]
    distances = results["distances"][0]

    print(f"\nTop-{TOP_K} retrieved chunks:")
    total_retrieved_chars = 0
    for j, (doc, dist) in enumerate(zip(retrieved_docs, distances)):
        total_retrieved_chars += len(doc)
        print(f"  [{j+1}] distance={dist:.4f} | {len(doc)} chars")
        print(f"      {doc[:150]}...")

    coverage = total_retrieved_chars / len(doc_text) * 100
    print(f"\nRetrieval coverage: {total_retrieved_chars}/{len(doc_text)} chars ({coverage:.1f}%)")

    retrieval_results.append({
        "sample_idx": i,
        "question": question,
        "gold_answer": gold_answer,
        "doc_length": len(doc_text),
        "num_chunks": len(chunks),
        "retrieved_text": "\n\n---\n\n".join(retrieved_docs),
        "distances": distances,
        "coverage_pct": coverage,
    })


Sample 1: 1389 chunks from 555451 chars
Question: how many segmented sales did the 5 largest customers account for in 2008?
Gold answer: $ 4236 million


/Users/christopherstewart/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz



Top-5 retrieved chunks:
  [1] distance=0.5875 | 500 chars
      has	caused,	our	customers	to	maintain	tighter inventory	management.	We	experienced	this	beginning	in	the	fourth	quarter	of	2008	and	expect	this	trend	...
  [2] distance=0.5899 | 500 chars
      	sales	to	the	segment's	five	largest	customers accounted	for	approximately	42%	of	the	segment's	net	sales.	Besides	selling	directly	to	carriers	and	op...
  [3] distance=0.6083 | 500 chars
      n	period.

Due	to	the	nature	of	the	segment's	business,	many	of	the	agreements	we	enter	into	are	long-term	contracts	that	require	sizeable	investments...
  [4] distance=0.6136 | 500 chars
      segment's	net	sales	in	2007.	Approximately	56%	of	the	segment's	net	sales	in	2008	were	to	markets	outside	the U.S.,	the	largest	of	which	were	Brazil,	...
  [5] distance=0.6748 | 500 chars
      End	Customer

|  | 2008 | 2007 | 2006 |
| :--- | :--- | :--- | :--- |
| United\tStates | 49% | 51% | 44% |
| Latin\tAmerica | 14% | 12% | 10% |
| Euro...

Retr

## Retrieval Diagnostics

Summarize retrieval behavior across the 5 samples. Are we seeing the "all or nothing" pattern?

In [12]:
print(f"{'Sample':<8} {'Doc chars':<12} {'Chunks':<8} {'Coverage %':<12} {'Min dist':<10} {'Max dist':<10}")
print("-" * 60)
for r in retrieval_results:
    print(
        f"{r['sample_idx']+1:<8} "
        f"{r['doc_length']:<12} "
        f"{r['num_chunks']:<8} "
        f"{r['coverage_pct']:<12.1f} "
        f"{min(r['distances']):<10.4f} "
        f"{max(r['distances']):<10.4f}"
    )

print("\n--- Interpretation ---")
print("If distances are very similar across chunks, the embeddings may not")
print("differentiate relevant from irrelevant content for these long documents.")
print("If coverage is very high, retrieval is essentially returning the whole doc.")
print("If coverage is very low but distances are high, retrieval found nothing useful.")

Sample   Doc chars    Chunks   Coverage %   Min dist   Max dist  
------------------------------------------------------------
1        555451       1389     0.5          0.5875     0.6748    
2        271894       680      0.9          0.7392     1.2139    
3        1440541      3602     0.2          0.6173     0.7942    
4        397044       993      0.6          0.5338     0.8025    
5        372954       933      0.7          0.7356     0.9494    

--- Interpretation ---
If distances are very similar across chunks, the embeddings may not
differentiate relevant from irrelevant content for these long documents.
If coverage is very high, retrieval is essentially returning the whole doc.
If coverage is very low but distances are high, retrieval found nothing useful.


## Generation with Gemini 2.5 Flash

For each sample, pass the retrieved context + question to Gemini and compare the generated answer with the gold answer.

In [13]:
from google import genai

gemini_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

PROMPT_TEMPLATE = """You are a financial analyst. Answer the following question using ONLY the provided context. If the context does not contain enough information, say "INSUFFICIENT CONTEXT".

Context:
{context}

Question: {question}

Answer concisely."""

generation_results = []

for r in retrieval_results:
    prompt = PROMPT_TEMPLATE.format(
        context=r["retrieved_text"],
        question=r["question"],
    )

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    generated = response.text.strip()

    print(f"\n{'=' * 80}")
    print(f"Sample {r['sample_idx']+1}")
    print(f"Question:  {r['question']}")
    print(f"Gold:      {r['gold_answer']}")
    print(f"Generated: {generated}")
    print(f"Coverage:  {r['coverage_pct']:.1f}%")

    generation_results.append({
        **r,
        "generated_answer": generated,
        "prompt_length": len(prompt),
    })


Sample 1
Question:  how many segmented sales did the 5 largest customers account for in 2008?
Gold:      $ 4236 million
Generated: In 2008, sales to the segment's top five customers represented approximately 45% of the segment's net sales.
Coverage:  0.5%

Sample 2
Question:  what was the percent of the total contractual payment obligations that was associated with operating lease obligations
Gold:      20.2%
Generated: INSUFFICIENT CONTEXT
Coverage:  0.9%

Sample 3
Question:  what is the growth rate in net revenue in 2008?
Gold:      -3.2%
Generated: Net revenue in 2007 was $1,297.6 million.
Net revenue in 2008 was $833.8 million.

The growth rate is calculated as ((Net Revenue 2008 - Net Revenue 2007) / Net Revenue 2007) * 100.

Growth Rate = (($833.8 million - $1,297.6 million) / $1,297.6 million) * 100
Growth Rate = (-$463.8 million / $1,297.6 million) * 100
Growth Rate = -0.3574291 * 100
Growth Rate = -35.74%

The growth rate in net revenue in 2008 was -35.74%.
Coverage:  0.2%

S

## Summary

In [15]:
print(f"{'#':<4} {'Coverage%':<11} {'Prompt len':<12} {'Gold answer':<30} {'Generated (first 60 chars)':<60}")
print("-" * 120)
for g in generation_results:
    print(
        f"{g['sample_idx']+1:<4} "
        f"{g['coverage_pct']:<11.1f} "
        f"{g['prompt_length']:<12} "
        f"{str(g['gold_answer'])[:28]:<30} "
        f"{g['generated_answer'][:60]:<60}"
    )

print("\n--- Notes ---")
print("- High coverage % suggests retrieval is returning most of the document")
print("- Low coverage % with wrong answers suggests retrieval missed the relevant passage")
print("- 'INSUFFICIENT CONTEXT' responses confirm the retrieval gap")
print("- Check if ChromaDB's default embedding model struggles with financial/tabular content")

#    Coverage%   Prompt len   Gold answer                    Generated (first 60 chars)                                  
------------------------------------------------------------------------------------------------------------------------
1    0.5         2814         $ 4236 million                 In 2008, sales to the segment's top five customers represent
2    0.9         2859         20.2%                          INSUFFICIENT CONTEXT                                        
3    0.2         2788         -3.2%                          Net revenue in 2007 was $1,297.6 million.
Net revenue in 200
4    0.6         2872         2.58                           INSUFFICIENT CONTEXT                                        
5    0.7         2816         2.6%                           To determine the percentage change in the balance of goodwil

--- Notes ---
- High coverage % suggests retrieval is returning most of the document
- Low coverage % with wrong answers suggests retrieval missed